In [1]:
# Flight source comparisons


In [2]:
from datetime import date, datetime, timedelta
from itertools import combinations

import polars as pl

from data_engineering.flights.flight_type import get_flights
from flights.evaluation.american_evals import add_airspace_columns, any_us_faa_airspace_expr
from flights.flights_comparison import df_flights_comparision, df_flights_comparison_stats

target_date = date(2026, 3, 1)
start_dt = datetime(target_date.year, target_date.month, target_date.day)
end_dt = start_dt + timedelta(days=1)


Loading airports from /Volumes/T2-SSD/planequery/data/raw/ourairports/airports.csv


In [3]:
flight_sources = {
    "algorithm_adsblol": get_flights(target_date, adsb_src="adsblol"),
    "algorithm_adsbx": get_flights(target_date, adsb_src="adsbx"),
    "algorithm_opensky": get_flights(target_date, adsb_src="opensky"),
    "opensky": get_flights(target_date, algorithm="opensky"),
    "adsbx": get_flights(target_date, algorithm="adsbx"),
}


In [4]:
pl.DataFrame(
    {
        "source": source,
        "n_flights": df.height,
        "n_icaos": df.get_column("icao").n_unique(),
        "null_first_lat": df.get_column("first_lat").null_count() if "first_lat" in df.columns else df.height,
        "null_last_lat": df.get_column("last_lat").null_count() if "last_lat" in df.columns else df.height,
        "null_takeoff_airport_ident": df.get_column("takeoff_airport_ident").null_count(),
        "null_landing_airport_ident": df.get_column("landing_airport_ident").null_count(),
    }
    for source, df in flight_sources.items()
)


source,n_flights,n_icaos,null_first_lat,null_last_lat,null_takeoff_airport_ident,null_landing_airport_ident
str,i64,i64,i64,i64,i64,i64
"""algorithm_adsblol""",119196,48307,0,0,0,0
"""algorithm_adsbx""",145765,57168,0,0,15012,15012
"""algorithm_opensky""",106931,43044,0,0,6215,6215
"""opensky""",102695,39357,102695,102695,25926,18716
"""adsbx""",84096,33077,84096,84096,0,0


In [5]:
correct_source_flights = {
    "algorithm_adsblol": get_flights(target_date, adsb_src="adsblol"),
    "opensky": get_flights(target_date, algorithm="opensky"),
    "adsbx": get_flights(target_date, algorithm="adsbx"),
}

pl.DataFrame(
    {
        "source": source,
        "n_flights": df.height,
        "null_takeoff_airport_ident": df.get_column("takeoff_airport_ident").null_count(),
        "null_landing_airport_ident": df.get_column("landing_airport_ident").null_count(),
        "null_either_airport_ident": df.filter(
            pl.col("takeoff_airport_ident").is_null()
            | pl.col("landing_airport_ident").is_null()
        ).height,
        "null_both_airport_ident": df.filter(
            pl.col("takeoff_airport_ident").is_null()
            & pl.col("landing_airport_ident").is_null()
        ).height,
    }
    for source, df in correct_source_flights.items()
).with_columns(
    (pl.col("null_takeoff_airport_ident") / pl.col("n_flights")).alias("null_takeoff_pct"),
    (pl.col("null_landing_airport_ident") / pl.col("n_flights")).alias("null_landing_pct"),
    (pl.col("null_either_airport_ident") / pl.col("n_flights")).alias("null_either_pct"),
)


source,n_flights,null_takeoff_airport_ident,null_landing_airport_ident,null_either_airport_ident,null_both_airport_ident,null_takeoff_pct,null_landing_pct,null_either_pct
str,i64,i64,i64,i64,i64,f64,f64,f64
"""algorithm_adsblol""",119196,0,0,0,0,0.0,0.0,0.0
"""opensky""",102695,25926,18716,38910,5732,0.252456,0.182248,0.378889
"""adsbx""",84096,0,0,0,0,0.0,0.0,0.0


In [6]:
faa_airspace_count_sources = {
    "algorithm_adsblol": flight_sources["algorithm_adsblol"],
    "algorithm_opensky": flight_sources["algorithm_opensky"],
    "algorithm_adsbx": flight_sources["algorithm_adsbx"],
    "opensky": flight_sources["opensky"],
    "adsbx": flight_sources["adsbx"],
}

pl.DataFrame(
    {
        "source": source,
        "algorithm": "algorithm" if source.startswith("algorithm_") else source,
        "adsb_src": source.removeprefix("algorithm_") if source.startswith("algorithm_") else None,
        "n_flights": df.height,
        "n_flights_any_us_faa_airspace": add_airspace_columns(df)
        .filter(any_us_faa_airspace_expr())
        .height,
    }
    for source, df in faa_airspace_count_sources.items()
).with_columns(
    (pl.col("n_flights_any_us_faa_airspace") / pl.col("n_flights")).alias("any_us_faa_airspace_pct")
)


source,algorithm,adsb_src,n_flights,n_flights_any_us_faa_airspace,any_us_faa_airspace_pct
str,str,str,i64,i64,f64
"""algorithm_adsblol""","""algorithm""","""adsblol""",119196,67464,0.565992
"""algorithm_opensky""","""algorithm""","""opensky""",106931,56436,0.52778
"""algorithm_adsbx""","""algorithm""","""adsbx""",145765,70462,0.483395
"""opensky""","""opensky""",null,102695,53282,0.518837
"""adsbx""","""adsbx""",null,84096,50434,0.599719
